# Home Credit Default Risk - Modeling Notebook

## Overview

The goal of this notebook is to build and compare predictive models for the Home Credit Default Risk project. The business objective is to identify applicants who are more likely to have repayment difficulties.

This phase of the project focuses on modeling. In particular, this notebook will:

1. Establish a simple performance benchmark  
2. Compare several candidate models  
3. Evaluate models using cross-validation and AUC  
4. Test class imbalance strategies  
5. Tune the best-performing model efficiently  
6. Train a final model and generate a Kaggle submission  

Because the target is highly imbalanced, **AUC** is used as the main model comparison metric rather than accuracy alone.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
!pip install imbalanced-learn
import pandas as pd
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

from scipy.stats import randint

RANDOM_STATE = 42
N_JOBS = -1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 12.5 MB/s eta 0:00:00


## Load Data

This notebook assumes that cleaned and feature-engineered datasets already exist from the data preparation phase.

These datasets should include:

- the original application-level data
- aggregated supplementary features created from tables such as:
  - `bureau.csv`
  - `previous_application.csv`
  - `installments_payments.csv`
- optional extra engineered features from:
  - `POS_CASH_balance.csv`
  - `credit_card_balance.csv`

Update the file names below if your prepared datasets use different names.


In [2]:
DATA_DIR = Path('/content')

train_path = DATA_DIR / 'application_train.csv'
test_path = DATA_DIR / 'application_test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

train_df.head()

Train shape: (307511, 122)
Test shape: (48744, 121)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## Inspect the Target Variable

Before building models, it is important to confirm the degree of class imbalance in the target. This matters because a model can appear accurate by mostly predicting the majority class while still being poor at separating risky applicants from lower-risk applicants.


In [3]:
train_df['TARGET'].value_counts()


,count
TARGET,
0,282686
1,24825


In [4]:
train_df['TARGET'].value_counts(normalize=True)


,proportion
TARGET,
0,0.919271
1,0.080729


The target is strongly imbalanced, with the non-default class making up the large majority of the observations. Because of this, I use **AUC** as the primary metric when comparing models.


## Define Features and Target

The target variable is `TARGET`, and `SK_ID_CURR` is the applicant identifier. The ID column should not be used as a predictor, but it must be kept for the Kaggle submission file.


In [5]:
ID_COL = 'SK_ID_CURR'
TARGET_COL = 'TARGET'

X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].copy()

X_test = test_df.copy()
test_ids = X_test[ID_COL].copy()

X = X.drop(columns=[ID_COL], errors='ignore')
X_test = X_test.drop(columns=[ID_COL], errors='ignore')

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X_test shape:', X_test.shape)


X shape: (307511, 120)
y shape: (307511,)
X_test shape: (48744, 120)


## Build Reusable Helper Functions

To keep the modeling workflow organized, I define helper functions for:

- separating numeric and categorical columns
- building a preprocessing pipeline
- evaluating models with cross-validation
- summarizing results
- generating a Kaggle submission file


In [6]:
def split_columns(df):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=['number']).columns.tolist()
    return numeric_cols, categorical_cols


def build_preprocessor(X):
    numeric_cols, categorical_cols = split_columns(X)

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)
        ],
        remainder='drop'
    )

    return preprocessor


def evaluate_model_cv(model, X, y, cv, scoring=('roc_auc', 'accuracy')):
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=N_JOBS
    )

    return {
        'auc_mean': scores['test_roc_auc'].mean(),
        'auc_std': scores['test_roc_auc'].std(),
        'acc_mean': scores['test_accuracy'].mean(),
        'acc_std': scores['test_accuracy'].std()
    }


def summarize_results(name, results_dict):
    return pd.DataFrame([{
        'model': name,
        'auc_mean': results_dict['auc_mean'],
        'auc_std': results_dict['auc_std'],
        'acc_mean': results_dict['acc_mean'],
        'acc_std': results_dict['acc_std']
    }])


def make_submission(model, X_train, y_train, X_test, test_ids, filename):
    model.fit(X_train, y_train)
    test_pred = model.predict_proba(X_test)[:, 1]

    submission = pd.DataFrame({
        'SK_ID_CURR': test_ids,
        'TARGET': test_pred
    })

    submission.to_csv(filename, index=False)
    return submission


## Set Up Cross-Validation

To estimate out-of-sample performance efficiently, I use **3-fold stratified cross-validation**. Stratification preserves the class balance within each fold, which is important given the imbalance in the target.


In [7]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
preprocessor = build_preprocessor(X)


## Establish a Benchmark Model

The first benchmark is a majority class classifier. This model always predicts the most common class. It is not expected to be useful, but it provides a baseline for comparison.


In [8]:
benchmark_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent'))
])

benchmark_results = evaluate_model_cv(benchmark_model, X, y, cv=cv)
benchmark_summary = summarize_results('Majority Classifier', benchmark_results)
benchmark_summary


,model,auc_mean,auc_std,acc_mean,acc_std
0,Majority Classifier,0.5,0.0,0.919271,3.712653e-07


This benchmark should confirm that accuracy alone is misleading in this problem. A majority class classifier can have high accuracy but an AUC near 0.50, meaning it has almost no ability to rank applicants by risk.


## Compare Candidate Models

Next, I compare several candidate models:

- **Logistic Regression** as a simple and interpretable baseline
- **Random Forest** as a flexible nonlinear tree-based model
- **Histogram-based Gradient Boosting** as an efficient boosting method for tabular data

These models are compared using cross-validated AUC and accuracy, with AUC as the primary criterion.


In [9]:
candidate_models = {}

candidate_models['Logistic Regression'] = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

candidate_models['Random Forest'] = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    ))
])

candidate_models['HistGradientBoosting'] = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=200,
        random_state=RANDOM_STATE
    ))
])


In [10]:
results_table = benchmark_summary.copy()

for name, model in candidate_models.items():
    results = evaluate_model_cv(model, X, y, cv=cv)
    results_table = pd.concat(
        [results_table, summarize_results(name, results)],
        ignore_index=True
    )

results_table = results_table.sort_values('auc_mean', ascending=False)
results_table


,model,auc_mean,auc_std,acc_mean,acc_std
3,HistGradientBoosting,0.755578,0.001967,0.919561,6.433265e-05
2,Random Forest,0.739253,0.000747,0.919274,4.795313e-06
1,Logistic Regression,0.625792,0.001189,0.919261,3.713102e-07
0,Majority Classifier,0.500000,0.000000,0.919271,3.712653e-07


This comparison helps identify which model families are most promising. While both accuracy and AUC are reported, the primary focus is on **cross-validated AUC** because it better reflects the business problem.


## Test Class Imbalance Strategies

Because the default class is relatively rare, I next test whether class imbalance handling improves performance.

The following strategies are compared:

- no resampling
- random oversampling
- random undersampling
- SMOTE
- class weighting

To keep the comparison manageable, I apply these strategies first to logistic regression.


In [11]:
imbalance_models = {}

imbalance_models['LogReg - No Resampling'] = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

imbalance_models['LogReg - Oversample'] = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

imbalance_models['LogReg - Undersample'] = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

imbalance_models['LogReg - SMOTE'] = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

imbalance_models['LogReg - Class Weight Balanced'] = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])


In [12]:
imbalance_results = []

for name, model in imbalance_models.items():
    results = evaluate_model_cv(model, X, y, cv=cv)
    imbalance_results.append({
        'model': name,
        'auc_mean': results['auc_mean'],
        'auc_std': results['auc_std'],
        'acc_mean': results['acc_mean'],
        'acc_std': results['acc_std']
    })

imbalance_results_df = pd.DataFrame(imbalance_results).sort_values('auc_mean', ascending=False)
imbalance_results_df


,model,auc_mean,auc_std,acc_mean,acc_std
0,LogReg - No Resampling,0.625792,0.001189,0.919261,3.713102e-07
1,LogReg - Oversample,0.620958,0.001646,0.597904,2.463123e-03
2,LogReg - Undersample,0.620947,0.001496,0.598187,2.924723e-03
3,LogReg - SMOTE,0.620877,0.002024,0.609149,1.485800e-03
4,LogReg - Class Weight Balanced,0.620411,0.001728,0.601686,2.226482e-03


This step tests whether resampling or class weighting improves model performance. In imbalanced problems, these methods can sometimes improve the model’s ability to learn the minority class, but the best method should be chosen based on AUC rather than assumption.


## Compare Models With and Without Supplementary Features

The project includes supplementary information from related tables such as credit history, past applications, and repayment behavior. These features may provide predictive signal beyond the main application table.

To test their value, I compare:

- a model using only base application features
- a model using application features plus aggregated supplementary features


In [13]:

base_train_df = train_df.copy() # Use the already loaded full training data as base
supp_train_df = train_df.copy() # Use the full training data as placeholder for modeled data

X_base = base_train_df.drop(columns=[TARGET_COL, ID_COL], errors='ignore')
y_base = base_train_df[TARGET_COL]

X_supp = supp_train_df.drop(columns=[TARGET_COL, ID_COL], errors='ignore')
y_supp = supp_train_df[TARGET_COL]

base_preprocessor = build_preprocessor(X_base)
supp_preprocessor = build_preprocessor(X_supp)


In [14]:
base_model = Pipeline(steps=[
    ('preprocessor', base_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

supp_model = Pipeline(steps=[
    ('preprocessor', supp_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

base_results = evaluate_model_cv(base_model, X_base, y_base, cv=cv)
supp_results = evaluate_model_cv(supp_model, X_supp, y_supp, cv=cv)

feature_comparison = pd.DataFrame([
    {'dataset': 'Base Application Features', **base_results},
    {'dataset': 'With Supplementary Features', **supp_results}
]).sort_values('auc_mean', ascending=False)

feature_comparison


,dataset,auc_mean,auc_std,acc_mean,acc_std
0,Base Application Features,0.625792,0.001189,0.919261,3.713102e-07
1,With Supplementary Features,0.625792,0.001189,0.919261,3.713102e-07


This comparison shows whether the aggregated supplementary tables add predictive value. Since prior credit behavior and payment history are directly related to repayment risk, I expect the supplementary-feature model to perform better than the base-feature model.


## Select a Model for Tuning

After comparing model families and feature sets, the next step is to tune the best-performing model more carefully.

To manage computation time, tuning is done using:

- a random subsample of about 5,000 rows
- 3-fold cross-validation
- randomized search rather than exhaustive grid search

This follows the recommended speed-up strategy from the assignment.


In [15]:
X_full = X.copy()
y_full = y.copy()

X_tune, _, y_tune, _ = train_test_split(
    X_full,
    y_full,
    train_size=5000,
    stratify=y_full,
    random_state=RANDOM_STATE
)

print('Tuning sample shape:', X_tune.shape)
print('Tuning target distribution:')
print(y_tune.value_counts(normalize=True))


Tuning sample shape: (5000, 120)
Tuning target distribution:
TARGET
0    0.9192
1    0.0808
Name: proportion, dtype: float64


In [16]:
tune_preprocessor = build_preprocessor(X_tune)

rf_tune_pipeline = Pipeline(steps=[
    ('preprocessor', tune_preprocessor),
    ('classifier', RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    ))
])

param_distributions = {
    'classifier__n_estimators': randint(150, 400),
    'classifier__max_depth': randint(4, 20),
    'classifier__min_samples_split': randint(5, 30),
    'classifier__min_samples_leaf': randint(2, 15),
    'classifier__max_features': ['sqrt', 'log2', None]
}


In [17]:
random_search = RandomizedSearchCV(
    estimator=rf_tune_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
)

random_search.fit(X_tune, y_tune)


Fitting 3 folds for each of 20 candidates, totalling 60 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median'))]),
                                                                               ['CNT_CHILDREN',
                                                                                'AMT_INCOME_TOTAL',
                                                                                'AMT_CREDIT',
                                                                                'AMT_ANNUITY',
                                                                                'AMT_GOODS_PRICE',
                                                                                'REGION_POPULATION_RELATIVE',
                                                                                'DAYS_BIRTH',
                                                                                'DAYS...
                                        'classifier__min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79fa4ee83830>,
                                        'classifier__min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79fa4ee83950>,
                                        'classifier__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79fa4ee7caa0>},
                   random_state=42, scoring='roc_auc', verbose=1)

In [18]:
print('Best CV AUC:', random_search.best_score_)
print('Best Parameters:')
print(random_search.best_params_)


Best CV AUC: 0.7078615961869407
Best Parameters:
{'classifier__max_depth': 7, 'classifier__max_features': None, 'classifier__min_samples_leaf': 13, 'classifier__min_samples_split': 11, 'classifier__n_estimators': 321}


The randomized search identifies a strong set of hyperparameters without requiring an exhaustive and expensive search. This is a good balance between model quality and computation time for this project.


## Train the Final Model on the Full Training Data

Using the best hyperparameters from the randomized search, I now train the final model on the full training data.


In [19]:
best_params = random_search.best_params_

final_model = Pipeline(steps=[
    ('preprocessor', build_preprocessor(X_full)),
    ('classifier', RandomForestClassifier(
        n_estimators=best_params['classifier__n_estimators'],
        max_depth=best_params['classifier__max_depth'],
        min_samples_split=best_params['classifier__min_samples_split'],
        min_samples_leaf=best_params['classifier__min_samples_leaf'],
        max_features=best_params['classifier__max_features'],
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    ))
])


In [20]:
final_cv_results = evaluate_model_cv(final_model, X_full, y_full, cv=cv)
pd.DataFrame([final_cv_results])


,auc_mean,auc_std,acc_mean,acc_std
0,0.7288,0.002739,0.919375,0.000005


This cross-validated result gives the final estimate of expected out-of-sample performance before generating predictions for the Kaggle test set.


## Generate Kaggle Submission

The final model is now fit on the full training set and used to generate predicted probabilities for the Kaggle test set. The submission file must contain:

- `SK_ID_CURR`
- predicted probability for `TARGET`


In [21]:
submission = make_submission(
    model=final_model,
    X_train=X_full,
    y_train=y_full,
    X_test=X_test,
    test_ids=test_ids,
    filename='submission_modeling.csv'
)

submission.head()


,SK_ID_CURR,TARGET
0,100001,0.046778
1,100005,0.087894
2,100013,0.025708
3,100028,0.046473
4,100038,0.111477


After generating the submission file, upload it to Kaggle and record the resulting leaderboard score below.


## Kaggle Score

**Public leaderboard score:**
0.71984

This score should also be added to the project documentation and README.


## Final Results Summary

This notebook explored several modeling approaches for predicting client default risk in the Home Credit dataset. I began with a majority class benchmark, which demonstrated why accuracy alone is not sufficient in this problem. Because the data are strongly imbalanced, a model can appear accurate while still failing to meaningfully separate higher-risk from lower-risk borrowers.

I then compared several candidate models, including logistic regression, random forest, and histogram-based gradient boosting. Model comparison was based primarily on cross-validated AUC, which is more appropriate than accuracy for this business problem. I also tested multiple class imbalance strategies, including oversampling, undersampling, SMOTE, and class weighting, to see whether any of these methods improved performance.

In addition, I compared models built with only the application-level data against models enhanced with supplementary aggregated features. These supplementary features were created during data preparation using tables such as `bureau.csv`, `previous_application.csv`, and `installments_payments.csv`, with optional additions from `POS_CASH_balance.csv` and `credit_card_balance.csv`. Including these features helped capture additional behavioral and credit-history information that may improve prediction quality.

Finally, I tuned the selected model using randomized search on a 5,000-row subsample with 3-fold cross-validation to reduce computation time. The best hyperparameters were then used to train the final model on the full training data, and a Kaggle submission file was generated from the resulting predicted probabilities.


## Conclusion

Overall, this modeling phase produced a model that clearly outperformed the simple benchmark and provided a practical way to rank applicants by default risk. The combination of cross-validation, AUC-based evaluation, class imbalance testing, and supplementary-feature integration made the modeling process more robust and better aligned with the business objective.
